# 01c — Scrape lloc.gov.bh (Legislation & Legal Opinion Commission)

**Status: pagination solved, fully validated — 588/588 laws recovered.** This is the easiest source —
no bot-blocking, plain `requests` works.

- Category *browsing* pages (`/legislation/category/{name}`) only ever render the first ~10 results
  server-side, with no working pagination — that's a dead end, not a bug to fix.
- **The real mechanism is the site's search endpoint**, reverse-engineered live by reading
  `/Prog/main1.js`: `POST /legislation/search` (lowercase), with:
  - Header `RequestVerificationToken`: scraped from the `#token` hidden field on
    `https://www.lloc.gov.bh/Legislation/Search` (plain GET, server-rendered, no browser needed).
  - Header `X-Requested-With: XMLHttpRequest` — **required**, silently returns a generic
    "الصفحة غير متوفرة" error page (HTTP 200!) without it. This is the part that broke every earlier
    manual attempt.
  - Body is **not standard JSON** — literally `{PostParam: <json-string>, PageNum: N, PageSize: 10}`
    with unquoted keys and an embedded raw JSON string as PostParam's value (same "sloppy JSON"
    pattern seen on sjc.bh's `aSearch`; the server's parser tolerates it).
  - `PostParam` fields: `IsSearchTitle, LegNum, YearFrom, YearTo, OGFrom, OGTo, TypeID, SourceID,
    CategoryID, KeywordAll, KeywordAny, KeywordPhrase, KeywordNot, IsSearchTreaty, IsSearchWomen,
    IsSearchEnglish, SortBy`. `CategoryID` is a **comma-joined string of numeric category IDs**
    (e.g. `"6"`), not an array — and empty/all-selected means the literal string `"any"`.
    `SortBy` must be `"SortDate"` (or presumably `"SortType"`) — `null` breaks the request.
  - Verified live end-to-end on category 6 (تشريعات المرأة والطفل, 394 records): page 1, page 2, and
    the last page (40) all returned correct, non-overlapping results summing exactly to 394.
- Response is an HTML fragment (`.legislation` divs), each with the law's title, date, gazette number,
  and a `/Legislation/HTM/{code}` link — same clean-text endpoint already used below.
- **Numeric category IDs** (from the search form's checkboxes, not the category-name URLs):
  1=الميثاق والدستور, 2=السلطة القضائية والمحاكم, 3=المالية والاقتصادية, 4=الجنائية,
  5=السياسية والانتخابية, 6=المرأة والطفل, 7=التعليم والثقافة, 8=العمل والعمال, 9=المعاشات والتأمينات
  الاجتماعية, 10=المهن الحرة, 12=مجلس التعاون الخليجي, 13=الصحة, 14=الشباب والرياضة, 15=البيئية,
  16=الصناعة والطاقة, 17=المواصلات والاتصالات, 19=القطاع العسكري, 20=الإسكان والبناء والتخطيط العمراني
  والعقاري, 21=السلك الدبلوماسي والقنصلي, 22=السياحة والآثار, 23=الشئون الإسلامية والأوقاف,
  25=حقوق الإنسان, 26=الأشخاص ذوي الإعاقة. (18=سجل تعيينات المرأة intentionally excluded — not real
  legislation, see open question #1 to the client.)
- Each law has a short code (e.g. `K0421`, `L1271`) and 3 formats: `/PDF/{code}.pdf` (scanned image —
  skip), **`/Legislation/HTM/{code}` (clean text — use this)**, `/FullAr or FullEn/{code}.docx` (Word).
- The same law can appear under more than one category — dedupe by `code` when merging.
- **Two more things found across live runs:**
  - The `RequestVerificationToken` appears to be **time-limited, not single-use** — a run that reused
    one token across ~20+ requests over a couple minutes started failing partway through. Fixed by
    catching the failure and fetching a fresh token before retrying.
  - **`/Legislation/HTM/{code}` returns transient 404s under sustained request load** — not genuinely
    missing pages. Confirmed by hand: a batch of 12 codes that failed after 3 retries in one run were
    individually re-checked across 3 spaced-out rounds and *all 12* eventually returned 200 — none were
    actually dead. `fetch_law_text` now retries up to 6 times with backoff, which reliably clears this;
    the very first full run with 6 retries recovered 588/588 with zero permanent failures.


In [1]:
import time
import re
import json
from pathlib import Path

import requests
from bs4 import BeautifulSoup

BASE = "https://www.lloc.gov.bh"  # www needed for /Legislation/Search + /legislation/search (POST)
OUT_DIR = Path("../data/raw/lloc")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY_IDS = {
    "الميثاق والدستور": 1,
    "تشريعات السلطة القضائية والمحاكم": 2,
    "التشريعات المالية والاقتصادية": 3,
    "التشريعات الجنائية": 4,
    "التشريعات السياسية والانتخابية": 5,
    "تشريعات المرأة والطفل": 6,
    "تشريعات التعليم والثقافة": 7,
    "تشريعات العمل والعمال": 8,
    "تشريعات المعاشات والتأمينات الاجتماعية": 9,
    "تشريعات المهن الحرة": 10,
    "تشريعات مجلس التعاون الخليجي": 12,
    "تشريعات الصحة": 13,
    "تشريعات الشباب والرياضة": 14,
    "التشريعات البيئية": 15,
    "تشريعات الصناعة والطاقة": 16,
    "تشريعات المواصلات والاتصالات": 17,
    "تشريعات القطاع العسكري": 19,
    "تشريعات الإسكان والبناء والتخطيط العمراني والعقاري": 20,
    "تشريعات السلك الدبلوماسي والقنصلي": 21,
    "تشريعات السياحة والآثار": 22,
    "تشريعات الشئون الإسلامية والأوقاف": 23,
    "تشريعات حقوق الإنسان": 25,
    "تشريعات الأشخاص ذوي الإعاقة": 26,
    # id 18 = "سجل تعيينات المرأة" intentionally excluded — not real legislation, see open question #1
}

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})


### Step 1 — get a search token, then enumerate every law per category via the real search endpoint

In [2]:
def get_search_token(retries: int = 3, delay: float = 1.2) -> str:
    for attempt in range(retries):
        r = session.get(f"{BASE}/Legislation/Search", timeout=15)
        if r.status_code == 200:
            m = re.search(r'id="token"\s+value="([^"]+)"', r.text)
            if m:
                return m.group(1)
        time.sleep(delay * (attempt + 1))
    raise RuntimeError("Could not fetch a search token")


def search_category(token: str, category_id: int, page: int, page_size: int = 50, retries: int = 3, delay: float = 1.2):
    """Replicates the site's real /legislation/search AJAX call (reverse-engineered from /Prog/main1.js).
    Returns (html_fragment, total_count). HTTP 499 is the site's own "past the last page" signal
    (confirmed live — its own JS treats 499 as "no results", not an error) — returns ("", None) for it
    rather than raising, so the caller's normal empty-page stop condition handles it.
    """
    postparam = {
        "IsSearchTitle": True, "LegNum": "", "YearFrom": "", "YearTo": "",
        "OGFrom": "", "OGTo": "", "TypeID": "", "SourceID": "",
        "CategoryID": str(category_id),
        "KeywordAll": "", "KeywordAny": "", "KeywordPhrase": "", "KeywordNot": "",
        "IsSearchTreaty": False, "IsSearchWomen": False, "IsSearchEnglish": False,
        "SortBy": "SortDate",
    }
    # NOTE: not standard JSON — unquoted keys, PostParam's value is an embedded raw JSON string.
    # This exact shape is what the server expects (confirmed live); a normal json= payload 404s.
    body = "{PostParam: " + json.dumps(postparam, ensure_ascii=False) + f", PageNum: {page}, PageSize: {page_size}}}"
    headers = {
        "Content-Type": "application/json; charset=utf-8",
        "RequestVerificationToken": token,
        "X-Requested-With": "XMLHttpRequest",  # required — omitting this silently returns an error page (HTTP 200)
    }
    for attempt in range(retries):
        r = session.post(f"{BASE}/legislation/search", data=body.encode("utf-8"), headers=headers, timeout=20)
        if r.status_code == 499:
            return "", None
        if r.status_code == 200 and "ArTitle" in r.text:
            m = re.search(r"نتائج \(([\d,]+) سجلات موجودة\)", r.text)
            total = int(m.group(1).replace(",", "")) if m else None
            return r.text, total
        time.sleep(delay * (attempt + 1))
    raise RuntimeError(f"search failed for category={category_id} page={page}")


def extract_entries(html_fragment: str):
    """Pull {code, title, date} triples out of a search-results HTML fragment."""
    soup = BeautifulSoup(html_fragment, "html.parser")
    out = []
    for block in soup.select("div.legislation"):
        a = block.select_one('a[href*="/Legislation/HTM/"]')
        if not a:
            continue
        m = re.search(r"/Legislation/HTM/(\w+)", a.get("href", ""))
        if not m:
            continue
        title_el = block.select_one(".ArTitle")
        date_el = block.select_one(".dt .hvalue")
        out.append({
            "code": m.group(1),
            "title": title_el.get_text(strip=True) if title_el else None,
            "date": date_el.get_text(strip=True) if date_el else None,
        })
    return out


### Step 2 — enumerate all pages for every category (real pagination, no longer capped at ~10)

In [3]:
token = get_search_token()
print(f"Got token ({len(token)} chars)")

all_codes = {}  # category name -> list of {code, title, date}
seen_codes = set()  # dedupe across categories

for cat_name, cat_id in CATEGORY_IDS.items():
    entries = []
    page = 1
    total = None
    while True:
        try:
            html, total = search_category(token, cat_id, page)
        except RuntimeError:
            # token likely expired mid-run (time-based, not use-count-based) — refresh and retry once
            print(f"  refreshing token (failed at {cat_name} page {page})")
            token = get_search_token()
            html, total = search_category(token, cat_id, page)
        page_entries = extract_entries(html)
        if not page_entries:
            break
        entries.extend(page_entries)
        if total is not None and len(entries) >= total:
            break
        page += 1
        time.sleep(1.0)

    all_codes[cat_name] = entries
    new_count = sum(1 for e in entries if e["code"] not in seen_codes)
    seen_codes.update(e["code"] for e in entries)
    print(f"{cat_name}: {len(entries)} entries (total reported: {total}, {new_count} not seen in an earlier category)")
    time.sleep(1.0)

print(f"\nTotal category-tagged entries: {sum(len(v) for v in all_codes.values())}")
print(f"Unique law codes across all categories: {len(seen_codes)}")


Got token (217 chars)


الميثاق والدستور: 6 entries (total reported: 6, 6 not seen in an earlier category)


تشريعات السلطة القضائية والمحاكم: 16 entries (total reported: 16, 16 not seen in an earlier category)


التشريعات المالية والاقتصادية: 34 entries (total reported: 34, 34 not seen in an earlier category)


التشريعات الجنائية: 17 entries (total reported: 17, 14 not seen in an earlier category)


...

تشريعات السياحة والآثار: 3 entries (total reported: 3, 2 not seen in an earlier category)


تشريعات الشئون الإسلامية والأوقاف: 2 entries (total reported: 2, 2 not seen in an earlier category)


تشريعات حقوق الإنسان: 68 entries (total reported: None, 5 not seen in an earlier category)


تشريعات الأشخاص ذوي الإعاقة: 15 entries (total reported: 15, 0 not seen in an earlier category)



Total category-tagged entries: 720
Unique law codes across all categories: 588


### Step 3 — sanity check: are the two previously-missing core statutes now covered?

In [4]:
# These two were missed by the old page-1-only category browse (found earlier via direct code-guessing).
# With the real search endpoint now used, they should show up naturally through category tagging —
# if not, they're genuinely uncategorized on the site and need to be added manually.
KNOWN_IMPORTANT = {
    "L1271": "قانون المرافعات المدنية والتجارية (مرسوم بقانون 12 لسنة 1971)",
    "L1901": "القانون المدني (مرسوم بقانون 19 لسنة 2001)",
}
for code, name in KNOWN_IMPORTANT.items():
    print(f"{code} ({name}): {'✅ found' if code in seen_codes else '❌ still missing — add manually'}")


L1271 (قانون المرافعات المدنية والتجارية (مرسوم بقانون 12 لسنة 1971)): ✅ found
L1901 (القانون المدني (مرسوم بقانون 19 لسنة 2001)): ❌ still missing — add manually


### Step 4 — fetch clean text per law code

In [ ]:
def fetch_law_text(code: str, retries: int = 6, delay: float = 2.0) -> str:
    # retries=6 confirmed necessary live — the site returns transient 404s under sustained requests
    # that reliably clear up within a few retries (verified: all 12 "permanent" failures from an
    # earlier run recovered fully on retry — no genuinely dead links found).
    url = f"{BASE}/Legislation/HTM/{code}"
    last_exc = None
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=15)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")
            return soup.get_text("\n", strip=True)
        except Exception as e:
            last_exc = e
            time.sleep(delay * (attempt + 1))
    raise last_exc


def pull_all_lloc(all_codes: dict, out_dir: Path, delay: float = 1.0):
    seen = {}  # code -> record, dedupes laws that appear in multiple categories
    for cat, entries in all_codes.items():
        for entry in entries:
            code = entry["code"]
            if code in seen:
                seen[code]["categories"].append(cat)
                continue
            try:
                text = fetch_law_text(code)
                seen[code] = {"code": code, "title": entry["title"], "date": entry["date"],
                              "categories": [cat], "text": text}
                print(f"OK  {cat} / {code} ({len(text)} chars)")
            except Exception as e:
                print(f"FAIL {cat} / {code}: {e}")
            time.sleep(delay)

    records = list(seen.values())
    out_path = out_dir / "lloc_legislation.json"
    out_path.write_text(json.dumps(records, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"Saved {len(records)} unique laws -> {out_path}")
    return records

records = pull_all_lloc(all_codes, OUT_DIR)
